# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [16]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [17]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [18]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [19]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Nom exact (mapping)     : 17067 | restants : 833
[2] Fuzzy nom (mapping)     :   124 | restants : 709


In [20]:
# 1. Filtrer uniquement les joueurs matchés via la méthode Fuzzy
df_fuzzy_matches = df_final[df_final['match_method'].str.startswith('fuzzy_name', na=False)].copy()

# 2. Extraire le score numérique de la chaîne 'fuzzy_name(XX.X)' pour pouvoir trier
df_fuzzy_matches['fuzzy_score'] = df_fuzzy_matches['match_method'].str.extract(r'\((.*?)\)').astype(float)

# 3. Sélectionner et ordonner les colonnes pour une inspection visuelle confortable
# (Ajustez 'player_name' ou 'team' selon les vrais noms de colonnes de votre df_soccerdata)
cols_to_inspect = [
    'join_key',         # Le nom d'origine (SoccerData / FBref)
    'player',             # Le nom récupéré de Transfermarkt (df_tm)
    'fuzzy_score',      # Le score de similarité obtenu
    'season_year',      # La saison du match
    'tm_id'             # L'identifiant Transfermarkt associé
]

# Optionnel : Ajouter le club ou championnat si présent dans votre df_soccerdata (ex: 'team')
if 'team' in df_fuzzy_matches.columns:
    cols_to_inspect.insert(2, 'team')

# 4. Afficher les résultats triés du score le plus bas au plus haut 
# (C'est en bas de tableau que se cachent les potentielles erreurs de matching)
df_inspection = df_fuzzy_matches[cols_to_inspect].sort_values(by='fuzzy_score', ascending=True)

print("\nTop des matchs avec les scores les plus hauts")
print(df_inspection.tail(20).to_string(index=False))


Top des matchs avec les scores les plus hauts
            join_key               player            team  fuzzy_score  season_year     tm_id
     karl etta eyong      Karl Etta Eyong      Villarreal        100.0         2025 1038950.0
     karl etta eyong      Karl Etta Eyong         Levante        100.0         2025 1038950.0
         marc pubill          Marc Pubill Atlético Madrid        100.0         2025  844637.0
vladyslav krapyvtsov Vladyslav Krapyvtsov          Girona        100.0         2025 1169436.0
       jonathan rowe        Jonathan Rowe       Marseille        100.0         2025  579346.0
    hakon haraldsson     Hákon Haraldsson           Lille        100.0         2025  652275.0
       idrissa gueye        Idrissa Guèye            Metz        100.0         2025  126665.0
    merveille papela     Merveille Papela        Mainz 05        100.0         2020  405689.0
           keke topp            Keke Topp   Werder Bremen        100.0         2024  701757.0
      cassian

In [21]:
print("\nTop des matchs avec les scores les plus bas")
print(df_inspection.head(20).to_string(index=False))


Top des matchs avec les scores les plus bas
                    join_key                  player            team  fuzzy_score  season_year    tm_id
                  joe knight              Joe Knight        Brighton         85.7         2025 425026.0
              toni fernandez          Toni Fernández       Barcelona         85.7         2025 467271.0
              andres antanon          Andrés Antañón      Celta Vigo         85.7         2025 993722.0
        nikola krstovi u0107         Nikola Krstović        Atalanta         85.7         2025 124715.0
         bartosz bia u0142ek          Bartosz Białek       Wolfsburg         85.7         2020  24496.0
    vasilije ad u017ei u0107          Vasilije Adžić        Juventus         85.7         2025 423609.0
               antonio arena           Antonio Arena            Roma         85.7         2025 241889.0
          giuseppe ambrosino      Giuseppe Ambrosino          Napoli         85.7         2025 579746.0
    vasilije ad u01

Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [22]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

## **Les joueurs orphelins**

In [23]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [24]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 643
Taux joueurs orphelins : 10.38%


In [25]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season_year
2020      8
2021     26
2022     42
2023     72
2024     38
2025    523
dtype: int64


In [26]:
temp_data = still_missing[still_missing['season_year']<2025]
temp_data

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003.0,2020
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.000000,0.053613,0.000000,0.000000,cameron peupion,2002.0,2022
2,ENG-Premier League,2223,Leeds United,Wilfried Gnonto,ITA,"MF,FW",18,2003.0,24,14,...,NaN,0,NaN,NaN,NaN,NaN,NaN,wilfried gnonto,2003.0,2022
3,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.129427,0.129427,shea charles,2003.0,2022
4,ENG-Premier League,2223,Nottingham Forest,Alex Mighten,ENG,"FW,MF",20,2002.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,alex mighten,2002.0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,ITA-Serie A,2425,Lecce,Filip Marchwiński,POL,MF,22,2002.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,filip marchwi u0144ski,2002.0,2024
611,ITA-Serie A,2425,Lecce,Rareș-Cătălin Burnete,ROU,FW,20,2004.0,4,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,rare u0219 c u0103t u0103lin burnete,2004.0,2024
612,ITA-Serie A,2425,Milan,Álex Jiménez,ESP,"MF,DF",19,2005.0,22,14,...,NaN,0,NaN,NaN,NaN,NaN,NaN,alex jimenez,2005.0,2024
613,ITA-Serie A,2425,Torino,Alieu Njie,SWE,FW,19,2005.0,16,0,...,NaN,0,1.235164,0.560247,1.235164,1.752160,0.031047,alieu njie,2005.0,2024


In [27]:
df_tm

,player_id,valuation_season_year,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,...,url,current_club_domestic_competition_id,current_club_name,highest_market_value_in_eur,date,market_value_in_eur,player_club_domestic_competition_id,join_key,join_key_full,dob_key
0,3333,2019.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2020-04-08,6500000.0,GB1,james milner,james milner,1986
1,3333,2020.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2021-06-08,3000000.0,GB1,james milner,james milner,1986
2,3333,2021.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2022-06-15,2000000.0,GB1,james milner,james milner,1986
3,3333,2022.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2023-06-20,1500000.0,GB1,james milner,james milner,1986
4,3333,2023.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2024-05-27,1000000.0,GB1,james milner,james milner,1986
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17472,1275040,2025.0,Noham,Kamara,Noham Kamara,2025,1041,noham-kamara,France,Meaux,...,https://www.transfermarkt.co.uk/noham-kamara/p...,FR1,Olympique Lyonnais,1000000.0,2025-12-15,1000000.0,FR1,noham kamara,noham kamara,2007
17473,1296876,2024.0,Marc,Domènech,Marc Domènech,2024,237,marc-domenech,Spain,Llucmajor,...,https://www.transfermarkt.co.uk/marc-domenech/...,ES1,Real Club Deportivo Mallorca S.A.D.,1000000.0,2024-12-27,500000.0,NaN,marc domenech,marc domenech,2006
17474,1305792,2024.0,Álvaro,García Pascual,Álvaro García Pascual,2024,368,alvaro-garcia-pascual,Spain,Benalmádena,...,https://www.transfermarkt.co.uk/alvaro-garcia-...,ES1,Sevilla Fútbol Club S.A.D.,1000000.0,2025-03-27,250000.0,NaN,alvaro garcia pascual,alvaro garcia pascual,2002
17475,1390649,2024.0,Yan,Diomande,Yan Diomande,2025,23826,yan-diomande,Cote d'Ivoire,Abidjan,...,https://www.transfermarkt.co.uk/yan-diomande/p...,L1,RasenBallsport Leipzig,45000000.0,2025-06-09,1500000.0,L1,yan diomande,yan diomande,2006


In [28]:
df_soccerdata[df_soccerdata["player"] == "Álex Jiménez"]

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
2879,ENG-Premier League,2526,Bournemouth,Álex Jiménez,ESP,"DF,MF",20-361,2005.0,31,26,...,NaN,0,NaN,NaN,NaN,NaN,NaN,alex jimenez,2005.0,2025
15673,ITA-Serie A,2324,Milan,Álex Jiménez,ESP,DF,18,2005.0,3,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,alex jimenez,2005.0,2023
16306,ITA-Serie A,2425,Milan,Álex Jiménez,ESP,"MF,DF",19,2005.0,22,14,...,NaN,0,NaN,NaN,NaN,NaN,NaN,alex jimenez,2005.0,2024
16900,ITA-Serie A,2526,Milan,Álex Jiménez,ESP,DF,20-361,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,alex jimenez,2005.0,2025


In [29]:
df_mapping[df_mapping["PlayerFBref"].str.contains("Jiménez")]

df_mapping[df_mapping["tm_id"] == 741257]

,PlayerFBref,fbref_id,tm_id,TmPos,join_key
453,Alejandro Jimenez,faa13947,741257.0,Right-Back,alejandro jimenez


In [30]:
df_tm[df_tm["last_name"] == "Jiménez"]

,player_id,valuation_season_year,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,...,url,current_club_domestic_competition_id,current_club_name,highest_market_value_in_eur,date,market_value_in_eur,player_club_domestic_competition_id,join_key,join_key_full,dob_key
6231,206040,2019.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2020-04-08,40000000.0,GB1,raul jimenez,raul jimenez,1991
6232,206040,2020.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2021-06-08,28000000.0,GB1,raul jimenez,raul jimenez,1991
6233,206040,2021.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2022-06-15,18000000.0,GB1,raul jimenez,raul jimenez,1991
6234,206040,2022.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2023-06-20,6000000.0,GB1,raul jimenez,raul jimenez,1991
6235,206040,2023.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2024-05-27,5000000.0,GB1,raul jimenez,raul jimenez,1991
6236,206040,2024.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2025-05-30,5000000.0,GB1,raul jimenez,raul jimenez,1991
6237,206040,2025.0,Raúl,Jiménez,Raúl Jiménez,2025,931,raul-jimenez,Mexico,Tepeji del Río de Ocampo,...,https://www.transfermarkt.co.uk/raul-jimenez/p...,GB1,Fulham Football Club,50000000.0,2025-12-09,4000000.0,GB1,raul jimenez,raul jimenez,1991
16455,741257,2023.0,Álex,Jiménez,Álex Jiménez,2025,989,alex-jimenez,Spain,Leganés,...,https://www.transfermarkt.co.uk/alex-jimenez/p...,GB1,Association Football Club Bournemouth,4000000.0,2024-06-25,4000000.0,GB1,alex jimenez,alex jimenez,2005
